# Notebook 38 — The economic edge search: wins, losses, payoff ratios, and cross-sectional skill

**Objective:** Run a broad, controlled experiment instead of incrementally tuning one weak classifier. Compare eight trained model/target combinations, a random negative control, and an **unattainable oracle upper bound**. **Transaction costs are set to zero by design.** All realized outcomes are one-next-bar close-to-close returns, **not** simulated stop-loss/take-profit fills. The desired **2:1** ratio means average positive return divided by absolute average negative return >= 2.

**Guardrails:** Three chronological development folds, training-only preprocessing and return clipping, same observations/dates for every model, 20% final historical holdout excluded, prospective ledger untouched. The oracle is for diagnostic comparison ONLY.

In [ ]:
from pathlib import Path
import sys,time
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
assert (ROOT/'src').is_dir(), 'Run from repository root or notebooks/'
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.historical_market import add_historical_market_features
from stock_ml_dynamic.accuracy_features import add_accuracy_features
from stock_ml_dynamic.volume_regime_research import add_volume_regime_features
from stock_ml_dynamic.payoff_search import run_payoff_search,payoff_stats,MODELS
master,latest,bundle,manifest=load_master(ROOT/'artifacts/master_tables/1d')
print('Master:',master.shape,'tickers:',master.symbol.nunique(),'range:',master.bar_timestamp.min(),master.bar_timestamp.max())

## Research plan and preregistered settings

- **Eight trained candidates:** direction RF, direction ExtraTrees, positive-return HGB, positive-return logistic, return HGB, return RF, return ExtraTrees, return Ridge.
- **Controls:** randomized daily ranking (negative control) and perfect realized-return ranking (oracle, not tradable).
- **Daily selection:** rank all available names using *only predicted scores* and take top **5, 10, or 20**; equal weights, zero transaction costs.
- **Economic metrics:** win rate, average win, average loss, payoff ratio, profit factor, expectancy, compounded daily return, drawdown.
- **Ranking metrics:** daily cross-sectional Spearman IC, comparison against random, paired 10-day moving-block confidence intervals.
- **Interpretation:** multiple comparisons create selection bias. Winners of this notebook are hypotheses, not validated deployment models.

In [ ]:
SEED=42
TRAIN_CAP=60_000
TREES=180
POSITIONS=(5,10,20)
OUT=ROOT/'artifacts/payoff_search/1d'
print(dict(seed=SEED,training_cap=TRAIN_CAP,trees=TREES,positions=POSITIONS,transaction_cost_bps=0,model_count=len(MODELS)))

## Development-only feature construction

The historical final 20% is excluded before feature construction. Features and model preprocessing follow the existing research pipeline.

In [ ]:
eligible,eligibility=eligible_universe(master)
development,cutoff=development_holdout_split(eligible,.20)
expanded,historical=add_historical_market_features(development)
expanded,technical=add_accuracy_features(expanded)
expanded,volume=add_volume_regime_features(expanded)
feature_cols=list(dict.fromkeys(historical+technical+volume))
print('Development:',expanded.shape,'features:',len(feature_cols),'cutoff:',cutoff)
assert len(expanded)<len(master)

## Large paired model search — 24 trained fits, 30 ranking policies per fold

Eight fitted models × three folds = 24 fits; 10 model/control rankings × 3 position sizes × 3 folds = 90 evaluated portfolios. Model selection is *not* based on the closed holdout. RF/ExtraTrees are the expensive configurations.

In [ ]:
start=time.perf_counter()
res=run_payoff_search(expanded,feature_cols,folds=3,cap=TRAIN_CAP,trees=TREES,positions=POSITIONS,seed=SEED,output_dir=OUT)
print('Elapsed seconds:',round(time.perf_counter()-start,1),'saved to:',OUT)
display(res['audit']);display(res['timing'].round(5))

## Experiment A — Are winners large enough to offset losers?

**Payoff ratio** = mean positive trade return / absolute mean negative trade return. **Profit factor** = sum positive returns / absolute sum negative returns. A payoff ratio of 2:1 breaks even at 33.3% wins (zero costs), but a 2:1 **observed** ratio is not the same as placing a 2R target and 1R stop.

In [ ]:
summary=res['metrics'].groupby(['model','k'],as_index=False).agg(win_rate=('win_rate','mean'),avg_win=('avg_win','mean'),avg_loss=('avg_loss','mean'),payoff_ratio=('payoff_ratio','mean'),profit_factor=('profit_factor','mean'),expectancy=('expectancy','mean'),mean_daily_return=('mean_daily_return','mean'),max_drawdown=('max_drawdown','mean'))
display(summary.sort_values(['k','expectancy'],ascending=[True,False]).round(5))
for k in POSITIONS:
    g=summary[summary.k==k].sort_values('expectancy')
    fig,ax=plt.subplots(figsize=(11,4));ax.barh(g.model,g.expectancy*100);ax.axvline(0,color='gray',linewidth=1);ax.set_title(f'Top-{k}: average gross return per trade (%)');plt.tight_layout();plt.show()

## Experiment B — Is the edge consistent in all chronological folds?

A strong mean with one exceptional fold is insufficient. Compare each model's **three** fold outcomes, win/loss ratios, and drawdowns.

In [ ]:
for k in POSITIONS:
    g=res['metrics'].query('k==@k')
    print('Top',k,'gross daily return by fold')
    display(g.pivot(index='model',columns='fold',values='mean_daily_return').round(5))
    print('Payoff ratio by fold');display(g.pivot(index='model',columns='fold',values='payoff_ratio').round(3))
    print('Profit factor by fold');display(g.pivot(index='model',columns='fold',values='profit_factor').round(3))

## Experiment C — Does each model beat random ranking and the direction baseline?

The random control is a single seeded realization, not a confidence bound for randomness. The oracle represents the **maximum obtainable same-day ranking using the answers**, not a model. Inspect paired confidence intervals; no single 95% CI resolves the multiple-testing problem.

In [ ]:
for k in POSITIONS:
    print('Top',k,'paired mean daily lift vs direction RF')
    g=res['paired'].query('k==@k').groupby('challenger').agg(mean_lift=('mean_daily_lift','mean'),folds_positive=('mean_daily_lift',lambda s:int((s>0).sum())),min_ci_low=('ci_low','min'),max_ci_high=('ci_high','max'))
    display(g.sort_values('mean_lift',ascending=False).round(6))
print('Random and oracle diagnostics (not eligible for model promotion):')
display(summary[summary.model.isin(['random_control','oracle_upper_bound'])].round(5))

## Experiment D — Daily cross-sectional information coefficient

If a regressor has negative R² but positive **daily rank IC**, it can still be a useful stock selector. The random-control IC should be near zero. Compare IC across all three folds and all eight trained models.

In [ ]:
display(res['ic'].pivot(index='model',columns='fold',values='mean_daily_spearman_ic').round(5))
display(res['ic'].groupby('model').agg(mean_ic=('mean_daily_spearman_ic','mean'),positive_ic_fraction=('ic_positive_fraction','mean')).sort_values('mean_ic',ascending=False).round(5))
fig,ax=plt.subplots(figsize=(11,4));res['ic'].groupby('model').mean_daily_spearman_ic.mean().sort_values().plot.barh(ax=ax);ax.axvline(0,color='gray');ax.set_title('Mean daily cross-sectional rank IC');plt.tight_layout();plt.show()

## Experiment E — Why win rate alone can mislead

Inspect the trade-return distribution and the contribution of extreme winners and losers. **No future outcome is used to select trades**; realized outcomes are used here only for diagnostics.

In [ ]:
k=10
for name in ['direction_rf','return_hgb','return_rf','return_extra_trees','positive_hgb','random_control']:
    g=res['trades'].query('model==@name and k==@k').next_return
    print(name,'N',len(g),'win',round((g>0).mean(),3),'mean',round(g.mean(),5),'median',round(g.median(),5),'p01/p05/p95/p99',np.round(g.quantile([.01,.05,.95,.99]).to_numpy(),4),'top 1% contribution',round(g.nlargest(max(1,int(len(g)*.01))).sum()/len(g),5))
fig,ax=plt.subplots(figsize=(11,4))
for name in ['direction_rf','return_hgb','return_rf','random_control']:
    vals=res['trades'].query('model==@name and k==10').next_return.clip(-.08,.08)
    ax.hist(vals,bins=60,histtype='step',density=True,label=name)
ax.legend();ax.set_title('Selected-trade return distributions, clipped for display only');plt.tight_layout();plt.show()

## Experiment F — Economic frontier and fold-robust shortlist

Require an average gross expectancy above the random control, positive mean daily IC, and positive expectancy in at least 2/3 folds. This is a **research filter**, not a valid significance test or a tuned trading rule. The oracle is excluded.

In [ ]:
rows=[]
for (name,k),g in res['metrics'].groupby(['model','k']):
    if name in ['random_control','oracle_upper_bound']:continue
    random_mean=res['metrics'].query('model=="random_control" and k==@k').expectancy.mean()
    mean_ic=res['ic'].query('model==@name').mean_daily_spearman_ic.mean()
    rows.append(dict(model=name,k=k,mean_expectancy=g.expectancy.mean(),mean_payoff=g.payoff_ratio.mean(),mean_profit_factor=g.profit_factor.mean(),positive_folds=int((g.expectancy>0).sum()),lift_vs_random=g.expectancy.mean()-random_mean,mean_ic=mean_ic,passes_screen=bool((g.expectancy>0).sum()>=2 and g.expectancy.mean()>random_mean and mean_ic>0)))
frontier=pd.DataFrame(rows).sort_values(['passes_screen','mean_expectancy'],ascending=False)
display(frontier.round(5));frontier.to_csv(OUT/'frontier.csv',index=False)

## Interpretation and next research decision

1. **If all models are near random:** Stop architecture tinkering; prioritize feature informativeness, alternative horizons, regime conditioning, and data quality. A 42% three-class accuracy is not itself proof of skill without matching the class base rate and proper controls.
2. **If regression ranks outperform:** Explore ranking objectives (pairwise or listwise), not only R² optimization. Verify rank IC stability by fold and stock.
3. **If profit factor > 1 but payoff ratio < 2:** The model can still be profitable; the proposed 2:1 criterion is a design goal, not a mathematical prerequisite.
4. **If no model meets 2:1:** Do **not** manufacture it by assuming stops and targets fill. A true +2R/−1R backtest requires entry/exit timing and intraday OHLC high/low sequencing rules.
5. **If the oracle strongly beats models:** This establishes theoretical selection opportunity, **not** that available features can predict it.

**Caution:** Top-k ranks require knowing the full daily candidate set and obtaining a feasible fill after features become available. Daily-close features paired with close-to-next-close returns can introduce execution-timing optimism. This notebook is a gross-return research diagnostic, not an executable backtest.

In [ ]:
print('Gross-cost research completed. Models:',len(MODELS),'folds:',res['audit'].fold.nunique(),'policies:',len(res['metrics']))
print('Saved:')
for file in sorted(OUT.glob('*.csv')): print(' ',file.name)
print('Shortlist:');display(frontier[frontier.passes_screen].head(12).round(5))